# DCLP5 Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# Set up file paths
dclp5_data_path = "../../data/raw/DCLP5_Dataset_2022-01-20-5e0f3b16-c890-4ace-9e3b-531f3687cf53/"
output_path = "../../data/user_data_expansion/"

# Read the participant roster
roster_file = os.path.join(dclp5_data_path, "PtRoster.txt")
print(f"Reading {roster_file}")
roster_df = pd.read_csv(roster_file, delimiter="|")
print(f"Roster shape: {roster_df.shape}")
print(f"Columns: {roster_df.columns.tolist()}")
roster_df.head()

In [ ]:
# Read the insulin data to identify delivery device type
insulin_file = os.path.join(dclp5_data_path, "Insulin.txt")
print(f"Reading {insulin_file}")
insulin_df = pd.read_csv(insulin_file, delimiter="|")
print(f"Insulin data shape: {insulin_df.shape}")
print(f"Columns: {insulin_df.columns.tolist()}")
insulin_df.head()

In [ ]:
# Examine insulin delivery routes
print("Unique insulin delivery routes:")
print(insulin_df['InsRoute'].value_counts())
print("\nSample data for each route type:")
print(insulin_df.groupby('InsRoute')[['PtID', 'ParentInsulinListID']].head(3))

In [ ]:
# Create a function to determine primary insulin delivery device per patient
def get_primary_insulin_delivery(patient_data):
    """
    Determine the primary insulin delivery device for a patient.
    Logic: If a patient has any pump usage, they are classified as 'Pump', 
    otherwise 'Injection'
    """
    if 'Pump' in patient_data['InsRoute'].values:
        return 'Pump'
    else:
        return 'Injection'

# Group by PtID and determine primary delivery method
patient_delivery = insulin_df.groupby('PtID').apply(get_primary_insulin_delivery)
patient_delivery_df = patient_delivery.reset_index()
patient_delivery_df.columns = ['PtID', 'insulin_delivery_device']

print(f"Patient delivery device summary:")
print(patient_delivery_df['insulin_delivery_device'].value_counts())
patient_delivery_df.head(10)

In [ ]:
# Create the final dataframe with one row per PtID
# Start with the roster to ensure we have all participants
final_df = roster_df[['PtID', 'EnrollDt', 'RandDt', 'trtGroup', 'PtStatus', 'SiteID']].copy()

# Merge with insulin delivery device information
final_df = final_df.merge(patient_delivery_df, on='PtID', how='left')

# Fill any missing insulin delivery device info (though there shouldn't be any)
final_df['insulin_delivery_device'] = final_df['insulin_delivery_device'].fillna('Unknown')

print(f"Final dataframe shape: {final_df.shape}")
print(f"Missing insulin delivery device info: {final_df['insulin_delivery_device'].isna().sum()}")
print("\nInsulin delivery device distribution:")
print(final_df['insulin_delivery_device'].value_counts())
final_df.head()

In [ ]:
# Save the initial dataframe to CSV
output_file = os.path.join(output_path, "DCLP5.csv")
final_df.to_csv(output_file, index=False)
print(f"Saved dataframe to: {output_file}")
print(f"Initial dataframe shape: {final_df.shape}")
print(f"Columns: {final_df.columns.tolist()}")

In [ ]:
# Display summary statistics
print("="*50)
print("DCLP5 User Data Expansion Summary")
print("="*50)
print(f"Total participants: {len(final_df)}")
print(f"Participants by treatment group:")
print(final_df['trtGroup'].value_counts())
print(f"\nParticipants by insulin delivery device:")
print(final_df['insulin_delivery_device'].value_counts())
print(f"\nParticipants by status:")
print(final_df['PtStatus'].value_counts())

# Cross-tabulation
print(f"\nCross-tab: Treatment Group vs Insulin Delivery Device:")
crosstab = pd.crosstab(final_df['trtGroup'], final_df['insulin_delivery_device'])
print(crosstab)

In [ ]:
# Update insulin_delivery_device to be more specific (t:slim X2 pump was used in DCLP5)
final_df['insulin_delivery_device'] = final_df['insulin_delivery_device'].replace('Pump', 't:slim X2')

# Add insulin_delivery_algorithm column based on treatment group
# Note: DCLP5 was pediatric study comparing SAP vs Control-IQ
final_df['insulin_delivery_algorithm'] = final_df['trtGroup'].map({
    'SAP': 'basal-bolus',
    'CLC': 'Control-IQ'
})

print("Updated dataframe:")
print(f"Shape: {final_df.shape}")
print(f"Columns: {final_df.columns.tolist()}")
print("\nInsulin delivery device distribution:")
print(final_df['insulin_delivery_device'].value_counts())
print("\nInsulin delivery algorithm distribution:")
print(final_df['insulin_delivery_algorithm'].value_counts())
print("\nCross-tab: Treatment Group vs Algorithm:")
print(pd.crosstab(final_df['trtGroup'], final_df['insulin_delivery_algorithm']))
final_df.head()

In [ ]:
# Save the updated dataframe
output_file = os.path.join(output_path, "DCLP5.csv")
final_df.to_csv(output_file, index=False)
print(f"Updated dataframe saved to: {output_file}")
print(f"Final shape: {final_df.shape}")
print(f"Final columns: {final_df.columns.tolist()}")

In [ ]:
# Analyze CGM device usage from DCLP5 data files
# Read CGM data files to identify which patients used which CGM devices

# DexcomClarityCGM contains most patients
dexcom_clarity_file = os.path.join(dclp5_data_path, "DexcomClarityCGM.txt")
print(f"Reading {dexcom_clarity_file}")
dexcom_clarity_df = pd.read_csv(dexcom_clarity_file, delimiter="|")
dexcom_ptids = set(dexcom_clarity_df['PtID'].unique())
print(f"Patients with Dexcom Clarity data: {len(dexcom_ptids)}")

# OtherCGM contains fewer patients - likely backup/alternative readings
other_cgm_file = os.path.join(dclp5_data_path, "OtherCGM.txt")
other_cgm_df = pd.read_csv(other_cgm_file, delimiter="|")
other_cgm_ptids = set(other_cgm_df['PtID'].unique())
print(f"Patients with Other CGM data: {len(other_cgm_ptids)}")

# Total CGM coverage
all_cgm_ptids = dexcom_ptids.union(other_cgm_ptids)
print(f"Total patients with any CGM data: {len(all_cgm_ptids)} out of {len(final_df)}")

# Based on DCLP5 study timeframe (2019-2021) and being Dexcom-sponsored, 
# this would have used Dexcom G6 (which was the current model during study period)
print("\\nDCLP5 study period: 2019-2021 -> Dexcom G6 era")

In [ ]:
# Add cgm_device column based on analysis
# DCLP5 was conducted 2019-2021, during Dexcom G6 era

def assign_cgm_device(ptid):
    """Assign CGM device based on data availability and study context"""
    if ptid in all_cgm_ptids:
        # DCLP5 used Dexcom G6 during 2019-2021 study period
        return "Dexcom G6"
    else:
        # Very few patients without CGM data - assign NaN
        return np.nan

# Apply CGM device assignment
final_df['cgm_device'] = final_df['PtID'].apply(assign_cgm_device)

print("CGM device distribution:")
print(final_df['cgm_device'].value_counts(dropna=False))
print(f"\nPatients with NaN CGM device: {final_df['cgm_device'].isna().sum()}")

# Show updated dataframe structure
print(f"\nUpdated dataframe shape: {final_df.shape}")
print(f"Columns: {final_df.columns.tolist()}")
final_df.head()

In [ ]:
# Save the dataframe with CGM device information
output_file = os.path.join(output_path, "DCLP5.csv")
final_df.to_csv(output_file, index=False)
print(f"Dataframe with CGM device info saved to: {output_file}")
print(f"Shape: {final_df.shape}")
print(f"Columns: {final_df.columns.tolist()}")

# Summary
print("\n" + "="*60)
print("DCLP5 USER DATA EXPANSION WITH CGM")
print("="*60)
print(f"Total participants: {len(final_df)}")
print(f"\nTreatment groups:")
print(final_df['trtGroup'].value_counts())
print(f"\nInsulin delivery device:")
print(final_df['insulin_delivery_device'].value_counts())
print(f"\nInsulin delivery algorithm:")
print(final_df['insulin_delivery_algorithm'].value_counts(dropna=False))
print(f"\nCGM device:")
print(final_df['cgm_device'].value_counts(dropna=False))

In [ ]:
# Read ethnicity and race data from DiabScreening file
screening_file = os.path.join(dclp5_data_path, "DiabScreening.txt")
print(f"Reading {screening_file}")
screening_df = pd.read_csv(screening_file, delimiter="|")

# Extract relevant columns for ethnicity analysis
ethnicity_data = screening_df[['PtID', 'Ethnicity', 'Race', 'RaceDs']].copy()

print("Ethnicity distribution:")
print(ethnicity_data['Ethnicity'].value_counts(dropna=False))
print("\\nRace distribution:")
print(ethnicity_data['Race'].value_counts(dropna=False))
print("\\nRace descriptions (for multi-race participants):")
print(ethnicity_data['RaceDs'].value_counts(dropna=False))

In [ ]:
# Create function to format ethnicity combining race and Hispanic/Latino status
def format_ethnicity(row):
    """Format ethnicity according to requirements"""
    race = row['Race']
    ethnicity = row['Ethnicity'] 
    race_desc = row['RaceDs']
    
    # Handle missing race data
    if pd.isna(race) or race == '':
        return np.nan
    
    # Start with race
    if race == 'More than one race':
        if pd.notna(race_desc) and race_desc.strip():
            # Use the detailed description, clean it up
            races = race_desc.replace(' and ', ', ').replace('and ', ', ')
            races = races.replace('caucasian', 'White').replace('hatian', 'Haitian')
            races = races.replace('asian', 'Asian').replace('Spanish', 'Hispanic/Latino')
            formatted_race = races
        else:
            formatted_race = 'Multiple races'
    else:
        formatted_race = race
    
    # Add Hispanic/Latino if applicable
    if pd.notna(ethnicity) and ethnicity == 'Hispanic or Latino':
        if 'Hispanic/Latino' not in formatted_race:
            formatted_race += ', Hispanic/Latino'
    
    return formatted_race

# Apply ethnicity formatting
ethnicity_data['formatted_ethnicity'] = ethnicity_data.apply(format_ethnicity, axis=1)

# Show examples of formatting
print("Examples of ethnicity formatting:")
examples = ethnicity_data[['PtID', 'Race', 'Ethnicity', 'RaceDs', 'formatted_ethnicity']].head(20)
print(examples)

print("\nFormatted ethnicity distribution:")
print(ethnicity_data['formatted_ethnicity'].value_counts(dropna=False))

In [ ]:
# Merge ethnicity data with final dataframe
ethnicity_mapping = ethnicity_data[['PtID', 'formatted_ethnicity']].copy()
ethnicity_mapping.columns = ['PtID', 'ethnicity']

# Merge with final dataframe
final_df = final_df.merge(ethnicity_mapping, on='PtID', how='left')

print(f"Final dataframe shape after adding ethnicity: {final_df.shape}")
print("\nAll unique ethnicity values:")
unique_ethnicities = final_df['ethnicity'].value_counts(dropna=False)
print(unique_ethnicities)

print(f"\nTotal unique ethnicity categories: {len(unique_ethnicities)}")
print(f"Participants with missing ethnicity data: {final_df['ethnicity'].isna().sum()}")

# Show updated dataframe structure
print(f"\nFinal columns: {final_df.columns.tolist()}")
final_df.head()

In [ ]:
# Save the dataframe with ethnicity data
output_file = os.path.join(output_path, "DCLP5.csv")
final_df.to_csv(output_file, index=False)
print(f"Dataframe with ethnicity data saved to: {output_file}")

# Summary with ethnicity
print("\n" + "="*70)
print("DCLP5 USER DATA EXPANSION WITH ETHNICITY")
print("="*70)
print(f"Total participants: {len(final_df)}")
print(f"Total columns: {len(final_df.columns)}")
print(f"Columns: {final_df.columns.tolist()}")

print("\nTreatment groups:")
print(final_df['trtGroup'].value_counts())

print("\nInsulin delivery device:")
print(final_df['insulin_delivery_device'].value_counts())

print("\nInsulin delivery algorithm:")
print(final_df['insulin_delivery_algorithm'].value_counts(dropna=False))

print("\nCGM device:")
print(final_df['cgm_device'].value_counts(dropna=False))

print("\nEthnicity distribution:")
print(final_df['ethnicity'].value_counts(dropna=False))

In [ ]:
# Extract age of diagnosis data from screening file  
# DiagAge column contains the age of diabetes diagnosis
diagnosis_age_data = screening_df[['PtID', 'DiagAge']].copy()
diagnosis_age_data.columns = ['PtID', 'age_of_diagnosis']

# Merge with final dataframe
final_df = final_df.merge(diagnosis_age_data, on='PtID', how='left')

print("Age of diagnosis statistics:")
print(final_df['age_of_diagnosis'].describe())
print(f"\nMissing age of diagnosis data: {final_df['age_of_diagnosis'].isna().sum()}")

print("\nAge of diagnosis distribution:")
age_distribution = final_df['age_of_diagnosis'].value_counts().sort_index()
print(age_distribution)

print(f"\nFinal dataframe shape: {final_df.shape}")
print(f"Final columns: {final_df.columns.tolist()}")
final_df.head()

In [ ]:
# Save the dataframe with age of diagnosis
output_file = os.path.join(output_path, "DCLP5.csv")
final_df.to_csv(output_file, index=False)
print(f"Dataframe with age_of_diagnosis saved to: {output_file}")

# Complete summary
print("\n" + "="*75)
print("COMPLETE DCLP5 USER DATA EXPANSION SUMMARY")
print("="*75)
print(f"Total participants: {len(final_df)}")
print(f"Total columns: {len(final_df.columns)}")
print(f"All columns: {final_df.columns.tolist()}")

print("\nAge of diagnosis summary:")
print(f"  Range: {final_df['age_of_diagnosis'].min()} - {final_df['age_of_diagnosis'].max()} years")
print(f"  Mean: {final_df['age_of_diagnosis'].mean():.1f} years")
print(f"  Median: {final_df['age_of_diagnosis'].median():.1f} years")
print(f"  Missing values: {final_df['age_of_diagnosis'].isna().sum()}")

print("\nComplete variable summary:")
print(f"- Insulin delivery device: t:slim X2 ({len(final_df)} participants)")
print(f"- Insulin delivery algorithm: Control-IQ vs basal-bolus")
print(f"- CGM device: Dexcom G6")
print(f"- Ethnicity categories: {len(final_df['ethnicity'].unique())} unique values")
print(f"- Age of diagnosis: pediatric population")

In [ ]:
# Add is_pregnant column based on analysis of pregnancy test data
# DCLP5 was a pediatric study - pregnancy would be an exclusion criterion
# Check pregnancy test data to confirm

try:
    pregnancy_file = os.path.join(dclp5_data_path, "DiabPregnancyTest.txt")
    pregnancy_df = pd.read_csv(pregnancy_file, delimiter="|")
    print("Pregnancy test data:")
    print(pregnancy_df['PregnancyTestResult'].value_counts(dropna=False))
    
    # Analysis shows pregnancy exclusion for pediatric study
    final_df['is_pregnant'] = False
    print("\\nPregnancy status added: All participants have is_pregnant = False")
    print("This reflects pediatric study population with pregnancy exclusion criterion")
    
except Exception as e:
    print(f"Could not read pregnancy data: {e}")
    # For pediatric study, assume no pregnancy
    final_df['is_pregnant'] = False
    print("Added is_pregnant = False for all participants (pediatric study)")

print(f"\\nUpdated dataframe shape: {final_df.shape}")
print(f"is_pregnant column distribution:")
print(final_df['is_pregnant'].value_counts())

final_df.head()

In [ ]:
# Save the complete dataframe
output_file = os.path.join(output_path, "DCLP5.csv")
final_df.to_csv(output_file, index=False)
print(f"Complete dataframe saved to: {output_file}")

# Summary
print("\n" + "="*80)
print("DCLP5 USER DATA EXPANSION WITH PREGNANCY STATUS")
print("="*80)
print(f"Total participants: {len(final_df)}")
print(f"Total columns: {len(final_df.columns)}")
print("\nComplete column list:")
for i, col in enumerate(final_df.columns, 1):
    print(f"  {i:2d}. {col}")

print("\nFinal data completeness:")
print(f"- Insulin delivery device: 100% (all t:slim X2)")
print(f"- Insulin delivery algorithm: {((final_df['insulin_delivery_algorithm'].notna()).sum()/len(final_df)*100):.1f}%") 
print(f"- CGM device: {((final_df['cgm_device'].notna()).sum()/len(final_df)*100):.1f}%")
print(f"- Ethnicity: {((final_df['ethnicity'].notna()).sum()/len(final_df)*100):.1f}% complete")
print(f"- Age of diagnosis: {((final_df['age_of_diagnosis'].notna()).sum()/len(final_df)*100):.1f}% complete") 
print(f"- Pregnancy status: 100% (all False - pediatric exclusion criterion)")
print("\nDataset ready for analysis!")

In [ ]:
# Add insulin_delivery_modality column based on treatment group
final_df['insulin_delivery_modality'] = final_df['trtGroup'].map({
    'SAP': 'SAP',
    'CLC': 'AID'
})

print("Insulin delivery modality distribution:")
print(final_df['insulin_delivery_modality'].value_counts())
print(f"\nCross-tab: Treatment Group vs Insulin Delivery Modality:")
print(pd.crosstab(final_df['trtGroup'], final_df['insulin_delivery_modality']))

print(f"\nUpdated dataframe shape: {final_df.shape}")
print(f"Updated columns: {final_df.columns.tolist()}")
final_df.head()

In [ ]:
# Save the dataframe with insulin_delivery_modality
output_file = os.path.join(output_path, "DCLP5.csv")
final_df.to_csv(output_file, index=False)
print(f"Dataframe with insulin_delivery_modality saved to: {output_file}")

# Summary with delivery modality
print("\n" + "="*85)
print("DCLP5 USER DATA EXPANSION WITH DELIVERY MODALITY")
print("="*85)
print(f"Total participants: {len(final_df)}")
print(f"Total columns: {len(final_df.columns)}")
print("\nComplete column list:")
for i, col in enumerate(final_df.columns, 1):
    print(f"  {i:2d}. {col}")

print("\nInsulin delivery modality distribution:")
print(final_df['insulin_delivery_modality'].value_counts())
print("\nDataset ready for analysis!")

In [ ]:
# Analyze insulin types for bolus and basal delivery
print("Detailed analysis of insulin data:")
print(f"Total insulin records: {len(insulin_df)}")
print(f"Unique patients in insulin data: {len(insulin_df['PtID'].unique())}")

print("\nUnique insulin delivery routes:")
print(insulin_df['InsRoute'].value_counts())

print("\nColumns in insulin data:")
print(insulin_df.columns.tolist())

# Check insulin types
print("\nAll unique insulin types (ParentInsulinListID):")
insulin_types = insulin_df['ParentInsulinListID'].value_counts()
print(insulin_types)

# Classify insulin types as bolus vs basal based on known insulin characteristics
bolus_insulins = [
    'Novolog (Aspart)',
    'Humalog (Lispro)', 
    'Novolog Fiasp',
    'Regular (R) (Humulin R or Novolin R)',
    'Admelog'
]

basal_insulins = [
    'Lantus (Glargine) 2 times per day',
    'Lantus (Glargine) 1 time per day', 
    'Degludec (Tresiba)',
    'Toujeo (Glargine, U300)',
    'Basaglar (Glargine, U100)',
    'Levemir (Detemir) 1 time per day'
]

print(f"\nClassified bolus insulins: {bolus_insulins}")
print(f"Classified basal insulins: {basal_insulins}")

# Check for any unclassified insulins
all_insulins = set(insulin_df['ParentInsulinListID'].dropna().unique())
classified_insulins = set(bolus_insulins + basal_insulins)
unclassified = all_insulins - classified_insulins
if unclassified:
    print(f"Unclassified insulins: {unclassified}")
else:
    print("All insulins successfully classified!")

In [ ]:
# Create functions to determine PUMP-ONLY bolus and basal insulin types for each patient
def get_pump_insulin_types_for_patient(ptid, insulin_data):
    """Get bolus and basal insulin types for pump therapy only (exclude injections)"""
    # Filter to only pump insulin records
    patient_pump_insulin = insulin_data[(insulin_data['PtID'] == ptid) & (insulin_data['InsRoute'] == 'Pump')]
    
    bolus_insulins_found = []
    basal_insulins_found = []
    
    for _, row in patient_pump_insulin.iterrows():
        insulin_name = row['ParentInsulinListID']
        if pd.notna(insulin_name):
            if insulin_name in bolus_insulins:
                bolus_insulins_found.append(insulin_name)
            elif insulin_name in basal_insulins:
                basal_insulins_found.append(insulin_name)
    
    # Remove duplicates and join with semicolons if multiple
    bolus_unique = list(set(bolus_insulins_found))
    basal_unique = list(set(basal_insulins_found))
    
    bolus_result = '; '.join(bolus_unique) if bolus_unique else np.nan
    basal_result = '; '.join(basal_unique) if basal_unique else np.nan
    
    return bolus_result, basal_result

# Extract PUMP-ONLY insulin types for all patients
print("Extracting PUMP-ONLY insulin types for all patients...")
pump_insulin_type_results = []

for ptid in final_df['PtID']:
    bolus, basal = get_pump_insulin_types_for_patient(ptid, insulin_df)
    pump_insulin_type_results.append({
        'PtID': ptid,
        'insulin_type_bolus': bolus,
        'insulin_type_basal': basal
    })

pump_insulin_types_df = pd.DataFrame(pump_insulin_type_results)
print(f"Extracted PUMP-ONLY insulin types for {len(pump_insulin_types_df)} patients")

print("\nPump-only insulin distribution:")
print("Bolus insulin types (pump only):")
bolus_pump_counts = pump_insulin_types_df['insulin_type_bolus'].value_counts(dropna=False)
print(bolus_pump_counts)

print("\nBasal insulin types (pump only):")
basal_pump_counts = pump_insulin_types_df['insulin_type_basal'].value_counts(dropna=False)  
print(basal_pump_counts)

In [ ]:
# Merge with pump-only insulin types
final_df = final_df.merge(pump_insulin_types_df, on='PtID', how='left')

# For patients with pump insulin, if basal is missing, use the same insulin as bolus
# (since pumps use the same insulin for both bolus and basal)
final_df.loc[
    (final_df['insulin_delivery_device'] == 't:slim X2') & 
    (final_df['insulin_type_basal'].isna()) & 
    (final_df['insulin_type_bolus'].notna()),
    'insulin_type_basal'
] = final_df.loc[
    (final_df['insulin_delivery_device'] == 't:slim X2') & 
    (final_df['insulin_type_basal'].isna()) & 
    (final_df['insulin_type_bolus'].notna()),
    'insulin_type_bolus'
]

print(f"Updated dataframe shape: {final_df.shape}")
print(f"Updated columns: {final_df.columns.tolist()}")

print("\nUpdated insulin type distribution (PUMP THERAPY ONLY):")
print("Bolus insulin types:")
updated_bolus_counts = final_df['insulin_type_bolus'].value_counts(dropna=False)
print(updated_bolus_counts)

print("\nBasal insulin types:")
updated_basal_counts = final_df['insulin_type_basal'].value_counts(dropna=False)
print(updated_basal_counts)

print(f"\nMissing data check:")
print(f"Patients missing bolus insulin: {final_df['insulin_type_bolus'].isna().sum()}")
print(f"Patients missing basal insulin: {final_df['insulin_type_basal'].isna().sum()}")

final_df.head()

In [ ]:
# Save the dataframe with pump-only insulin types
output_file = os.path.join(output_path, "DCLP5.csv")
final_df.to_csv(output_file, index=False)
print(f"Dataframe with PUMP-ONLY insulin types saved to: {output_file}")

# Summary with insulin types
print("\n" + "="*80)
print("DCLP5 USER DATA EXPANSION WITH INSULIN TYPES")
print("="*80)
print(f"Total participants: {len(final_df)}")
print(f"Total columns: {len(final_df.columns)}")
print("\nAll columns:")
for i, col in enumerate(final_df.columns, 1):
    print(f"  {i:2d}. {col}")

print("\nInsulin delivery summary:")
print(f"- All {len(final_df)} participants use t:slim X2 insulin pump")
print(f"- All participants have pump-only insulin type data")

print("\nBolus insulin types:")
bolus_summary = final_df['insulin_type_bolus'].value_counts()
for insulin_type, count in bolus_summary.items():
    print(f"  {insulin_type}: {count} patients")

print("\nBasal insulin types:")  
basal_summary = final_df['insulin_type_basal'].value_counts()
for insulin_type, count in basal_summary.items():
    print(f"  {insulin_type}: {count} patients")

print("\nDataset complete with insulin type information!")

In [ ]:
# Final dataset cleanup: rename PtID to "id" and drop unnecessary columns
print("FINAL DATASET CLEANUP:")
print("="*50)

print("Before cleanup:")
print(f"Shape: {final_df.shape}")
print(f"Columns: {final_df.columns.tolist()}")

# Rename PtID to "id"
final_df = final_df.rename(columns={'PtID': 'id'})

# Drop specified columns
columns_to_drop = ['EnrollDt', 'RandDt', 'trtGroup', 'PtStatus', 'SiteID']
final_df = final_df.drop(columns=columns_to_drop)

print("\nAfter cleanup:")
print(f"Shape: {final_df.shape}")
print(f"Columns: {final_df.columns.tolist()}")

print("\nDropped columns:")
for col in columns_to_drop:
    print(f"  - {col}")

print("\nRenamed:")
print("  - PtID → id")

# Save the final cleaned dataset
output_file = os.path.join(output_path, "DCLP5.csv")
final_df.to_csv(output_file, index=False)
print(f"\nFinal cleaned dataset saved to: {output_file}")

# Show sample of final dataset
print("\nSample of final cleaned dataset:")
print(final_df.head())

print("\n" + "="*70)
print("FINAL DCLP5 USER DATA EXPANSION - COMPLETE")
print("="*70)
print(f"Total participants: {len(final_df)}")
print(f"Final columns ({len(final_df.columns)}): {final_df.columns.tolist()}")
print("\nDataset ready for analysis!")
print("✓ Comprehensive participant demographics and clinical data")
print("✓ Pump-only insulin types (study period only)")
print("✓ Clean column structure for analysis")